In [16]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient
from contants import openai_api_key

In [17]:
model_client = OpenAIChatCompletionClient(model="gpt-4o-2024-08-06", api_key=openai_api_key)

# Define a team.
assistant_agent = AssistantAgent(
    name="assistant_agent",
    system_message="You are a helpful assistant",
    model_client=model_client,
)
agent_team = RoundRobinGroupChat([assistant_agent], termination_condition=MaxMessageTermination(max_messages=2))

# Run the team and stream messages to the console.
stream = agent_team.run_stream(task="Write a beautiful poem 3-line about lake tangayika")

# Use asyncio.run(...) when running in a script.
await Console(stream)

# Save the state of the agent team.
team_state = await agent_team.save_state()


---------- TextMessage (user) ----------
Write a beautiful poem 3-line about lake tangayika


---------- TextMessage (assistant_agent) ----------
In Tanganyika's embrace, waters whisper tales,  
Ancient depths hold secrets, where serenity prevails,  
Sun-kissed ripples dance, as time gently sails.  


In [18]:
import json

## save state to disk

with open("team_state.json", "w") as f:
    json.dump(team_state, f)

## load state from disk
with open("team_state.json", "r") as f:
    team_state = json.load(f)

new_agent_team = RoundRobinGroupChat([assistant_agent], termination_condition=MaxMessageTermination(max_messages=2))
await new_agent_team.load_state(team_state)
stream = new_agent_team.run_stream(task="What was the last line of the poem you wrote?")
await Console(stream)
await model_client.close()


---------- TextMessage (user) ----------
What was the last line of the poem you wrote?
---------- TextMessage (assistant_agent) ----------
The last line of the poem I wrote was:  

"Sun-kissed ripples dance, as time gently sails."
